In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

In [2]:
import json
import re
import os
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

LABEL2ID = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

c:\Users\Therese\anaconda3\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


## 3. File paths

Change these paths if your data folder is different.

In [3]:
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

EVIDENCE_PATH = DATA_DIR / "evidence.json"
TRAIN_PATH = DATA_DIR / "train-claims.json"
DEV_PATH = DATA_DIR / "dev-claims.json"
TEST_PATH = DATA_DIR / "test-claims-unlabelled.json"

MODEL_NAME = "distilroberta-base"
MODEL_DIR = OUTPUT_DIR / "verifier_model"

DEV_PRED_PATH = OUTPUT_DIR / "dev_predictions.json"
TEST_PRED_PATH = OUTPUT_DIR / "test_predictions.json"

## 4. Utility functions

In [4]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def normalise_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%°]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def simple_tokenise(text: str) -> List[str]:
    return normalise_text(text).split()


def get_claim_items(claims_json: Dict) -> List[Tuple[str, Dict]]:
    return list(claims_json.items())


def concatenate_evidence(
    evidence_ids: List[str],
    evidence_corpus: Dict[str, str],
    max_evidence: int = 5
) -> str:
    selected_ids = evidence_ids[:max_evidence]
    evidence_texts = []

    for eid in selected_ids:
        if eid in evidence_corpus:
            evidence_texts.append(evidence_corpus[eid])

    if len(evidence_texts) == 0:
        return "No relevant evidence found."

    return " ".join(evidence_texts)

## 5. Load data

In [5]:
evidence_corpus = load_json(EVIDENCE_PATH)
train_claims = load_json(TRAIN_PATH)
dev_claims = load_json(DEV_PATH)

print("Number of evidence passages:", len(evidence_corpus))
print("Number of train claims:", len(train_claims))
print("Number of dev claims:", len(dev_claims))

Number of evidence passages: 1208827
Number of train claims: 1228
Number of dev claims: 154


## Build vocabulary

In [ ]:
def build_vocab(claims_json, evidence_corpus, min_freq=2, max_vocab_size=50000):
    counter = Counter()

    for claim_id, instance in claims_json.items():
        claim_text = instance["claim_text"]
        counter.update(simple_tokenise(claim_text))

        for eid in instance.get("evidences", []):
            if eid in evidence_corpus:
                counter.update(simple_tokenise(evidence_corpus[eid]))

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1,
        "<CLAIM>": 2,
        "<EVIDENCE>": 3
    }

    for word, freq in counter.most_common(max_vocab_size):
        if freq >= min_freq and word not in vocab:
            vocab[word] = len(vocab)

    return vocab


vocab = build_vocab(train_claims, evidence_corpus)
print("Vocab size:", len(vocab))# Retriever
class BM25Retriever:
    def __init__(self, evidence_corpus: Dict[str, str]):
        self.evidence_corpus = evidence_corpus
        self.evidence_ids = list(evidence_corpus.keys())
        self.evidence_texts = [evidence_corpus[eid] for eid in self.evidence_ids]

        print("Building BM25 index...")
        tokenised_corpus = [simple_tokenise(text) for text in tqdm(self.evidence_texts)]
        self.bm25 = BM25Okapi(tokenised_corpus)

    def retrieve(self, claim_text: str, top_k: int = 5) -> List[str]:
        query_tokens = simple_tokenise(claim_text)
        scores = self.bm25.get_scores(query_tokens)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [self.evidence_ids[i] for i in top_indices]

    def evaluate_recall_at_k(self, claims_json: Dict, k: int = 5) -> float:
        total = 0
        hit = 0

        for claim_id, instance in tqdm(get_claim_items(claims_json)):
            claim_text = instance["claim_text"]
            gold_evidence = set(instance.get("evidences", []))

            if len(gold_evidence) == 0:
                continue

            retrieved = set(self.retrieve(claim_text, top_k=k))

            if len(gold_evidence.intersection(retrieved)) > 0:
                hit += 1

            total += 1

        return hit / total if total > 0 else 0.0


Vocab size: 6872


## FAISS

In [24]:
import faiss
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer


class CachedFAISSDenseRetriever:
    def __init__(
        self,
        evidence_corpus,
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        cache_dir="outputs/faiss_cache",
        batch_size=64,
        device=None,
        force_rebuild=False
    ):
        self.evidence_corpus = evidence_corpus
        self.evidence_ids = list(evidence_corpus.keys())
        self.evidence_texts = [evidence_corpus[eid] for eid in self.evidence_ids]

        self.model_name = model_name
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)

        safe_model_name = model_name.replace("/", "_")
        self.index_path = self.cache_dir / f"{safe_model_name}.faiss"
        self.ids_path = self.cache_dir / f"{safe_model_name}_evidence_ids.json"

        self.model = SentenceTransformer(model_name, device=device)

        if self.index_path.exists() and self.ids_path.exists() and not force_rebuild:
            print("Loading cached FAISS index...")

            with open(self.ids_path, "r", encoding="utf-8") as f:
                cached_ids = json.load(f)

            if cached_ids != self.evidence_ids:
                print("Cached evidence IDs do not match current evidence.json. Rebuilding FAISS index...")
                self._build_and_save_index(batch_size)
            else:
                self.index = faiss.read_index(str(self.index_path))
                print("Cached FAISS index loaded.")

        else:
            print("No valid FAISS cache found. Building index...")
            self._build_and_save_index(batch_size)

    def _build_and_save_index(self, batch_size):
        evidence_embeddings = self.model.encode(
            self.evidence_texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        embedding_dim = evidence_embeddings.shape[1]

        # Inner product works as cosine similarity because embeddings are normalised.
        self.index = faiss.IndexFlatIP(embedding_dim)
        self.index.add(evidence_embeddings)

        faiss.write_index(self.index, str(self.index_path))

        with open(self.ids_path, "w", encoding="utf-8") as f:
            json.dump(self.evidence_ids, f, indent=2)

        print("Saved FAISS index to:", self.index_path)
        print("Saved evidence IDs to:", self.ids_path)

    def retrieve(self, claim_text, top_k=5):
        claim_embedding = self.model.encode(
            [claim_text],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        scores, indices = self.index.search(claim_embedding, top_k)

        return [self.evidence_ids[i] for i in indices[0]]

    def evaluate_recall_at_k(self, claims_json, k=5):
        total = 0
        hit = 0

        for claim_id, instance in tqdm(get_claim_items(claims_json)):
            claim_text = instance["claim_text"]
            gold_evidence = set(instance.get("evidences", []))

            if len(gold_evidence) == 0:
                continue

            retrieved = set(self.retrieve(claim_text, top_k=k))

            if len(gold_evidence.intersection(retrieved)) > 0:
                hit += 1

            total += 1

        return hit / total if total > 0 else 0.0

In [25]:
retriever = CachedFAISSDenseRetriever(
    evidence_corpus=evidence_corpus,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    cache_dir="outputs/faiss_cache",
    batch_size=64,
    device=device,
    force_rebuild=False
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5842.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


No valid FAISS cache found. Building index...


Batches: 100%|██████████| 18888/18888 [1:13:08<00:00,  4.30it/s]


Saved FAISS index to: outputs\faiss_cache\sentence-transformers_all-MiniLM-L6-v2.faiss
Saved evidence IDs to: outputs\faiss_cache\sentence-transformers_all-MiniLM-L6-v2_evidence_ids.json


## Encode text

In [7]:
def encode_tokens(tokens, vocab, max_len=512):
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
    return ids[:max_len]


def encode_claim_evidence(claim_text, evidence_text, vocab, max_len=512):
    tokens = (
        ["<CLAIM>"] 
        + simple_tokenise(claim_text) 
        + ["<EVIDENCE>"] 
        + simple_tokenise(evidence_text)
    )

    return encode_tokens(tokens, vocab, max_len=max_len)

## CNN + BiLSTM dataset

In [8]:
class CNNBiLSTMDataset(Dataset):
    def __init__(
        self,
        claims_json,
        evidence_corpus,
        vocab,
        max_len=512,
        max_evidence=5,
        use_gold_evidence=True,
        retriever=None,
        retrieval_top_k=5,
        is_test=False
    ):
        self.items = list(claims_json.items())
        self.evidence_corpus = evidence_corpus
        self.vocab = vocab
        self.max_len = max_len
        self.max_evidence = max_evidence
        self.use_gold_evidence = use_gold_evidence
        self.retriever = retriever
        self.retrieval_top_k = retrieval_top_k
        self.is_test = is_test

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        claim_id, instance = self.items[idx]
        claim_text = instance["claim_text"]

        if self.use_gold_evidence and not self.is_test:
            evidence_ids = instance.get("evidences", [])
        else:
            evidence_ids = self.retriever.retrieve(
                claim_text,
                top_k=self.retrieval_top_k
            )

        evidence_text = concatenate_evidence(
            evidence_ids=evidence_ids,
            evidence_corpus=self.evidence_corpus,
            max_evidence=self.max_evidence
        )

        input_ids = encode_claim_evidence(
            claim_text=claim_text,
            evidence_text=evidence_text,
            vocab=self.vocab,
            max_len=self.max_len
        )

        item = {
            "claim_id": claim_id,
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "evidence_ids": evidence_ids
        }

        if not self.is_test:
            item["label"] = torch.tensor(
                LABEL2ID[instance["claim_label"]],
                dtype=torch.long
            )

        return item


def cnn_bilstm_collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    input_ids = pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=vocab["<PAD>"]
    )

    attention_mask = (input_ids != vocab["<PAD>"]).long()

    output = {
        "claim_ids": [item["claim_id"] for item in batch],
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "evidence_ids": [item["evidence_ids"] for item in batch]
    }

    if "label" in batch[0]:
        output["labels"] = torch.stack([item["label"] for item in batch])

    return output

## CNN + BiLSTM model

In [9]:
class CNNBiLSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        cnn_channels=128,
        kernel_size=3,
        lstm_hidden_dim=128,
        lstm_layers=1,
        num_labels=4,
        dropout=0.3,
        pad_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_idx
        )

        self.conv = nn.Conv1d(
            in_channels=embedding_dim,
            out_channels=cnn_channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2
        )

        self.bilstm = nn.LSTM(
            input_size=cnn_channels,
            hidden_size=lstm_hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            lstm_hidden_dim * 2,
            num_labels
        )

    def forward(self, input_ids, attention_mask=None):
        # input_ids: [batch, seq_len]
        embedded = self.embedding(input_ids)
        # embedded: [batch, seq_len, embedding_dim]

        # Conv1d expects [batch, channels, seq_len]
        conv_input = embedded.transpose(1, 2)
        conv_output = F.relu(self.conv(conv_input))
        # conv_output: [batch, cnn_channels, seq_len]

        # LSTM expects [batch, seq_len, features]
        lstm_input = conv_output.transpose(1, 2)

        lstm_output, _ = self.bilstm(lstm_input)
        # lstm_output: [batch, seq_len, hidden_dim * 2]

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1)
            lstm_output = lstm_output.masked_fill(mask == 0, -1e9)

        pooled_output = torch.max(lstm_output, dim=1).values
        pooled_output = self.dropout(pooled_output)

        logits = self.classifier(pooled_output)

        return logits

## Training function

In [10]:
def train_cnn_bilstm(
    train_claims,
    dev_claims,
    evidence_corpus,
    vocab,
    epochs=5,
    batch_size=32,
    lr=1e-3,
    max_len=512,
    max_evidence=5,
    device="cpu"
):
    train_dataset = CNNBiLSTMDataset(
        claims_json=train_claims,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        use_gold_evidence=True,
        is_test=False
    )

    dev_dataset = CNNBiLSTMDataset(
        claims_json=dev_claims,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        use_gold_evidence=True,
        is_test=False
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=cnn_bilstm_collate_fn
    )

    dev_loader = DataLoader(
        dev_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=cnn_bilstm_collate_fn
    )

    model = CNNBiLSTMClassifier(
        vocab_size=len(vocab),
        embedding_dim=128,
        cnn_channels=128,
        kernel_size=3,
        lstm_hidden_dim=128,
        lstm_layers=1,
        num_labels=4,
        dropout=0.3,
        pad_idx=vocab["<PAD>"]
    ).to(device)

    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_macro_f1 = 0.0
    best_state_dict = None

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")

        model.train()
        total_loss = 0.0

        for batch in tqdm(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimiser.zero_grad()

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(logits, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimiser.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print("Training loss:", round(avg_loss, 4))

        dev_acc, dev_macro_f1 = evaluate_cnn_bilstm(
            model,
            dev_loader,
            device
        )

        print("Dev accuracy with gold evidence:", round(dev_acc, 4))
        print("Dev macro F1 with gold evidence:", round(dev_macro_f1, 4))

        if dev_macro_f1 > best_macro_f1:
            best_macro_f1 = dev_macro_f1
            best_state_dict = model.state_dict()
            print("New best model saved in memory.")

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model

## Evaluation function

In [11]:
def evaluate_cnn_bilstm(model, dataloader, device="cpu"):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    print(classification_report(
        all_labels,
        all_preds,
        target_names=[ID2LABEL[i] for i in range(4)]
    ))

    return acc, macro_f1

## Train CNN + BiLSTM

In [12]:
cnn_bilstm_model = train_cnn_bilstm(
    train_claims=train_claims,
    dev_claims=dev_claims,
    evidence_corpus=evidence_corpus,
    vocab=vocab,
    epochs=5,
    batch_size=32,
    lr=1e-3,
    max_len=512,
    max_evidence=5,
    device=device
)


Epoch 1/5


100%|██████████| 39/39 [00:11<00:00,  3.51it/s]


Training loss: 1.2629


100%|██████████| 5/5 [00:00<00:00, 26.79it/s]
c:\Users\Therese\anaconda3\envs\rag\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Therese\anaconda3\envs\rag\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Therese\anaconda3\envs\rag\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

                 precision    recall  f1-score   support

       SUPPORTS       0.51      0.79      0.62        68
        REFUTES       0.00      0.00      0.00        27
NOT_ENOUGH_INFO       0.48      0.56      0.52        41
       DISPUTED       0.00      0.00      0.00        18

       accuracy                           0.50       154
      macro avg       0.25      0.34      0.28       154
   weighted avg       0.35      0.50      0.41       154

Dev accuracy with gold evidence: 0.5
Dev macro F1 with gold evidence: 0.2844
New best model saved in memory.

Epoch 2/5


...

100%|██████████| 39/39 [00:09<00:00,  4.27it/s]


Training loss: 0.8058


100%|██████████| 5/5 [00:00<00:00, 24.80it/s]

                 precision    recall  f1-score   support

       SUPPORTS       0.55      0.87      0.67        68
        REFUTES       0.00      0.00      0.00        27
NOT_ENOUGH_INFO       0.62      0.71      0.66        41
       DISPUTED       0.00      0.00      0.00        18

       accuracy                           0.57       154
      macro avg       0.29      0.39      0.33       154
   weighted avg       0.41      0.57      0.47       154

Dev accuracy with gold evidence: 0.5714
Dev macro F1 with gold evidence: 0.3333
New best model saved in memory.



c:\Users\Therese\anaconda3\envs\rag\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Therese\anaconda3\envs\rag\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Therese\anaconda3\envs\rag\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shap

## Predict with retrieved evidence

In [13]:
def predict_claims_cnn_bilstm(
    claims_json,
    evidence_corpus,
    retriever,
    model,
    vocab,
    output_path,
    retrieval_top_k=10,
    max_evidence=3,
    batch_size=32,
    max_len=512,
    is_test=False,
    device="cpu"
):
    dataset = CNNBiLSTMDataset(
        claims_json=claims_json,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        use_gold_evidence=False,
        retriever=retriever,
        retrieval_top_k=retrieval_top_k,
        is_test=is_test
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=cnn_bilstm_collate_fn
    )

    model.eval()
    predictions = {}

    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            pred_ids = torch.argmax(logits, dim=1).cpu().numpy().tolist()

            for claim_id, pred_id, evidence_ids in zip(
                batch["claim_ids"],
                pred_ids,
                batch["evidence_ids"]
            ):
                predictions[claim_id] = {
                    "claim_label": ID2LABEL[pred_id],
                    "evidences": evidence_ids[:max_evidence]
                }

    save_json(predictions, output_path)
    print("Saved predictions to:", output_path)

    return predictions

## Generate dev predictions

In [26]:
CNN_BILSTM_DEV_PRED_PATH = OUTPUT_DIR / "dev_predictions_cnn_bilstm.json"

dev_predictions_cnn_bilstm = predict_claims_cnn_bilstm(
    claims_json=dev_claims,
    evidence_corpus=evidence_corpus,
    retriever=retriever,
    model=cnn_bilstm_model,
    vocab=vocab,
    output_path=CNN_BILSTM_DEV_PRED_PATH,
    retrieval_top_k=10,
    max_evidence=3,
    batch_size=32,
    max_len=512,
    is_test=False,
    device=device
)

  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:23<00:00,  4.73s/it]

Saved predictions to: outputs\dev_predictions_cnn_bilstm.json


In [27]:
!python eval.py --predictions outputs/dev_predictions_cnn_bilstm.json --groundtruth data/dev-claims.json

Evidence Retrieval F-score (F)    = 0.16097711811997525
Claim Classification Accuracy (A) = 0.45454545454545453
Harmonic Mean of F and A          = 0.2377538065270407
